In [101]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,cross_val_score,RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import joblib

In [ ]:
df = pd.read_csv('../data/loan_data.csv')

In [78]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
1,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
2,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
3,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
4,LP001013,Male,Yes,0,Not Graduate,No,2333,1516.0,95.0,360.0,1.0,Urban,Y


In [79]:
df.shape

(381, 13)

In [80]:
df.info()
#We will have to convert categorical columns into numerical columns
#We can see missing Values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            381 non-null    object 
 1   Gender             376 non-null    object 
 2   Married            381 non-null    object 
 3   Dependents         373 non-null    object 
 4   Education          381 non-null    object 
 5   Self_Employed      360 non-null    object 
 6   ApplicantIncome    381 non-null    int64  
 7   CoapplicantIncome  381 non-null    float64
 8   LoanAmount         381 non-null    float64
 9   Loan_Amount_Term   370 non-null    float64
 10  Credit_History     351 non-null    float64
 11  Property_Area      381 non-null    object 
 12  Loan_Status        381 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 38.8+ KB


In [81]:
#Handling Missing Value
df.isnull().sum()

,0
Loan_ID,0
Gender,5
Married,0
Dependents,8
Education,0
Self_Employed,21
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,0
Loan_Amount_Term,11


In [82]:
#Percentage of missing values
df.isnull().mean()*100

,0
Loan_ID,0.000000
Gender,1.312336
Married,0.000000
Dependents,2.099738
Education,0.000000
Self_Employed,5.511811
ApplicantIncome,0.000000
CoapplicantIncome,0.000000
LoanAmount,0.000000
Loan_Amount_Term,2.887139


In [83]:
#Dropping unnecessary columns
df = df.drop('Loan_ID',axis = 1) #We do not require the Loan ID for our prediction model


In [84]:
df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
1,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
2,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
3,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
4,Male,Yes,0,Not Graduate,No,2333,1516.0,95.0,360.0,1.0,Urban,Y


In [85]:
'''
Ways to handle missing values:
- Remove nan values
- Remove whole column
- Replace with the Mode/Median

Our dataset has only few missing values with low percentages.
For this project
we are going to replace the columns having alot of missing values we are going to replace it
with the mode of the column.
We are going to drop the Nan values for columns with less amount of missing values

'''
#Dropping nan values for columns with less missing values
df = df.dropna(subset = ['Gender','Dependents','Loan_Amount_Term'])




In [86]:
df.shape

(358, 12)

In [87]:
#Deciding whether to replace nan values with mode or mean for columns having more missing values
df['Self_Employed'].unique()
#It is categorival values so we are going to take mode
selfemployed_mode = df['Self_Employed'].mode()[0]

In [88]:
#Checking another column with higher missing values
df['Credit_History'].unique()
#Since we need a fractional point, we will take mode (1,0) --> ('Yes','No')
credithistory_mode = df['Credit_History'].mode()[0]

In [89]:
#Replacing the nan values
df['Self_Employed'].fillna(selfemployed_mode,inplace = True)
df['Credit_History'].fillna(credithistory_mode,inplace = True)

/tmp/ipython-input-4743349.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Self_Employed'].fillna(selfemployed_mode,inplace = True)
/tmp/ipython-input-4743349.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

In [90]:
#Now we have no missing values
df.isnull().sum()

,0
Gender,0
Married,0
Dependents,0
Education,0
Self_Employed,0
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,0
Loan_Amount_Term,0
Credit_History,0


In [91]:
#Converting Object datatype into Numerical datatype for Logistic Regression
print('Gender: ',df['Gender'].unique())
print('Married: ',df['Married'].unique())
print('Education: ',df['Education'].unique())
print('Self_Employed: ',df['Self_Employed'].unique())
print('Property_Area: ',df['Property_Area'].unique())
print('Loan_Status: ',df['Loan_Status'].unique())
print('Dependents: ',df['Dependents'].unique())


Gender:  ['Male' 'Female']
Married:  ['Yes' 'No']
Education:  ['Graduate' 'Not Graduate']
Self_Employed:  ['No' 'Yes']
Property_Area:  ['Rural' 'Urban' 'Semiurban']
Loan_Status:  ['N' 'Y']
Dependents:  ['1' '0' '2' '3+']


In [92]:
#For encoding our binary columns
binary_map = {
    'Gender': {'Male': 0, 'Female': 1},
    'Married': {'No': 0, 'Yes': 1},
    'Education': {'Not Graduate': 0, 'Graduate': 1},
    'Self_Employed': {'No': 0, 'Yes': 1},
    'Loan_Status': {'N': 0, 'Y': 1}  # target variable
}

for col, mapping in binary_map.items():
    df[col] = df[col].map(mapping)

In [93]:
#Property_Area; One-Hot Encoding (drop first to avoid multicollinearity)
df = pd.get_dummies(df, columns=['Property_Area'], drop_first=True,dtype = int)

In [94]:
'''
Dependents; handle '3+' and keep as numerical
Mapping '3+' to 3
We could create a seperate flag for 3plus, however for this project we are replacing it with 4
'''

df['Dependents'] = df['Dependents'].replace('3+', 3).astype(int)
df = pd.get_dummies(df, columns=['Dependents'], drop_first=True,dtype = int)


In [95]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 358 entries, 0 to 380
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Gender                   358 non-null    int64  
 1   Married                  358 non-null    int64  
 2   Education                358 non-null    int64  
 3   Self_Employed            358 non-null    int64  
 4   ApplicantIncome          358 non-null    int64  
 5   CoapplicantIncome        358 non-null    float64
 6   LoanAmount               358 non-null    float64
 7   Loan_Amount_Term         358 non-null    float64
 8   Credit_History           358 non-null    float64
 9   Loan_Status              358 non-null    int64  
 10  Property_Area_Semiurban  358 non-null    int64  
 11  Property_Area_Urban      358 non-null    int64  
 12  Dependents_1             358 non-null    int64  
 13  Dependents_2             358 non-null    int64  
 14  Dependents_3             358 no

In [96]:
df.head()

,Gender,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Property_Area_Semiurban,Property_Area_Urban,Dependents_1,Dependents_2,Dependents_3
0,0,1,1,0,4583,1508.0,128.0,360.0,1.0,0,0,0,1,0,0
1,0,1,1,1,3000,0.0,66.0,360.0,1.0,1,0,1,0,0,0
2,0,1,0,0,2583,2358.0,120.0,360.0,1.0,1,0,1,0,0,0
3,0,0,1,0,6000,0.0,141.0,360.0,1.0,1,0,1,0,0,0
4,0,1,0,0,2333,1516.0,95.0,360.0,1.0,1,0,1,0,0,0


In [98]:
#Dividing our dataset into dependent and independent features
x = df.drop('Loan_Status',axis = 1)
y = df['Loan_Status']

In [99]:
#Converting our columns into Standardscalar for better pattern recognition
num_cols = ['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term']
scaler = StandardScaler()
x[num_cols] = scaler.fit_transform(x[num_cols])

In [100]:
x.head()

,Gender,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area_Semiurban,Property_Area_Urban,Dependents_1,Dependents_2,Dependents_3
0,0,1,1,0,0.711630,0.092069,0.805980,0.285826,1.0,0,0,1,0,0
1,0,1,1,1,-0.398856,-0.539332,-1.350425,0.285826,1.0,0,1,0,0,0
2,0,1,0,0,-0.691384,0.447965,0.527735,0.285826,1.0,0,1,0,0,0
3,0,0,1,0,1.705666,-0.539332,1.258130,0.285826,1.0,0,1,0,0,0
4,0,1,0,0,-0.866761,0.095418,-0.341784,0.285826,1.0,0,1,0,0,0


In [103]:
#Validation Function
def evaluate_model(model):
  x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42) #Splitting our dataset for calidation
  model.fit(x_train,y_train)
  y_pred = model.predict(x_test) #Predicting our data
  accuracy = accuracy_score(y_test,y_pred) #Actual and predicted data
  cross_val = cross_val_score(model,x,y,cv=5)
  avg_cross_val = np.mean(cross_val)
  print(f'{model.__class__.__name__} - Accuracy: {accuracy: .2f}, Cross-Val-Score: {avg_cross_val: .2f}')
  return avg_cross_val

In [102]:
#Creating our models for our classification problem
models = {
    LogisticRegression(),
    svm.SVC()
}

In [104]:
model_score = {model.__class__.__name__:evaluate_model(model) for model in models}

RandomForestClassifier - Accuracy:  0.85, Cross-Val-Score:  0.83
SVC - Accuracy:  0.86, Cross-Val-Score:  0.84
DecisionTreeClassifier - Accuracy:  0.79, Cross-Val-Score:  0.77
LogisticRegression - Accuracy:  0.85, Cross-Val-Score:  0.84
GradientBoostingClassifier - Accuracy:  0.85, Cross-Val-Score:  0.81


In [108]:
#Tuning our model
def tune_model(model,param_grid):
  tuner = RandomizedSearchCV(model,param_grid,cv = 5, n_iter = 20, verbose = True, random_state = 42)
  tuner.fit(x,y)
  print(f"Best Score for {model.__class__.__name__}: {tuner.best_score_: .2f}")
  print(f"Best Parameter for {model.__class__.__name__}: {tuner.best_params_}")
  return tuner.best_estimator_

In [111]:
#Specify our parameters
log_reg_grid = {'C': np.logspace(-4,4,20), "solver": ["liblinear"]}
svc_grid = {'C' : [0.25,0.50,0.75,1], "kernel": ['linear']}


In [113]:
best_log_grid = tune_model(LogisticRegression(),log_reg_grid)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Score for LogisticRegression:  0.84
Best Parameter for LogisticRegression: {'solver': 'liblinear', 'C': np.float64(4.281332398719396)}


In [114]:
best_svm_grid = tune_model(svm.SVC(),svc_grid)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Score for SVC:  0.84
Best Parameter for SVC: {'kernel': 'linear', 'C': 0.25}


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=20. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [118]:
final_model = best_log_grid

In [ ]:
joblib.dump(final_model,'../models/loan_status_predictor.pkl')

In [ ]:
#Prediction System
sample_data = pd.DataFrame({
    'Gender': [1],
    'Married': [1],
    'Education': [0],
    'Self_Employed': [0],
    'ApplicantIncome': [1000],
    'CoapplicantIncome': [0.0],
    'LoanAmount': [150],
    'Loan_Amount_Term': [180],
    'Credit_History': [0],
    'Property_Area_Semiurban': [0],
    'Property_Area_Urban': [1],
    'Dependents_1': [0],
    'Dependents_2': [1],
    'Dependents_3': [0]
})

sample_data[num_cols] = scaler.transform(sample_data[num_cols])
loaded_model = joblib.load('../models/loan_status_predictor.pkl')
prediction = loaded_model.predict(sample_data)

result = "Loan Approved" if prediction[0] == 1 else "Loan Not Approved"
print(f"\nPrediction Result: {result}")

In [ ]:
joblib.dump(scaler,'../models/scaler.pkl')